## Universal Environment Setup

This first cell detects whether the notebook is running in **Google Colab** or **locally** (VS Code, Jupyter, PyCharm, terminal) and bootstraps the environment accordingly:

- **Colab**: clones the repository into `/content/`, installs `requirements.txt`, and sets the working directory.
- **Local**: traverses up to 6 directory levels looking for `requirements.txt` as the project-root marker.

After this cell runs, `ROOT` is set and `os.chdir(ROOT)` has been called, so every downstream path (CSV loads, model saves, output writes) is automatically relative to the project root.


In [ ]:
import os
import sys
from pathlib import Path

# ==========================================
# 1. ENVIRONMENT DETECTION & AUTO-SETUP
# ==========================================
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    # Scenario: Google Colab
    REPO_URL = "https://github.com/nassim0014/btc-llm-sentiment.git"
    REPO_NAME = "btc-llm-sentiment"
    COLAB_ROOT = Path('/content') / REPO_NAME

    if not COLAB_ROOT.exists():
        print(f"\U0001f680 Colab environment detected. Cloning repository...")
        !git clone {REPO_URL} /content/{REPO_NAME}
        print("\U0001f4e6 Installing dependencies...")
        !pip install -q -r /content/{REPO_NAME}/requirements.txt
    else:
        print(f"\u2705 Repository already exists in Colab.")
    ROOT = COLAB_ROOT
else:
    # Scenario: Local (VS Code, Jupyter, PyCharm, Terminal)
    current_dir = Path.cwd()
    ROOT = None

    # Bounded traversal: Look for 'requirements.txt' to find the project root
    for _ in range(6):
        if (current_dir / 'requirements.txt').exists():
            ROOT = current_dir
            break
        if current_dir == current_dir.parent:
            break
        current_dir = current_dir.parent

    if ROOT is None:
        raise FileNotFoundError(
            "\u274c Could not locate the project root (missing 'requirements.txt').\n"
            "If running locally, please ensure you have cloned the repo and opened this notebook from within the project directory."
        )

# ==========================================
# 2. FINALIZE PATHS & SETUP
# ==========================================
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

print(f"\u2705 Environment initialized. Root directory: {ROOT}")


# 01 — Data Loading

**Goal:** Fetch crypto news headlines and BTC-USD daily prices from reproducible remote sources, apply the two bug fixes (yfinance MultiIndex flattening + mixed date parsing), and persist a cleaned merged dataset for downstream notebooks.

**Author:** Nassim K.


## 1.1 Imports & global config


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import yfinance as yf
import requests
from tqdm.auto import tqdm

# Project root detection — works from notebook, script, or IDE
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'Data' / 'cryptonews.csv').exists() or (ROOT / 'notebooks').exists():
        break
    if ROOT == ROOT.parent:
        break
    ROOT = ROOT.parent

INTERIM = ROOT / 'notebooks' / 'interim'
INTERIM.mkdir(parents=True, exist_ok=True)
print(f'Project root: {ROOT}')
print(f'Interim dir : {INTERIM}')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1.2 Fetch crypto news from GitHub raw URL

We pull the news dataset live from the repository's `main` branch on GitHub. **No local CSV uploads** — this guarantees the notebook runs identically in Colab, Jupyter, VS Code, or CI.


In [ ]:
NEWS_URL = 'https://raw.githubusercontent.com/nassim0014/btc-llm-sentiment/main/Data/cryptonews.csv'
LOCAL_NEWS = ROOT / 'Data' / 'cryptonews.csv'

def fetch_news(url: str, fallback: Path) -> pd.DataFrame:
    """Fetch the news CSV from `url`, fall back to local copy if the request fails."""
    try:
        print(f'Downloading news from {url} ...')
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        from io import StringIO
        df = pd.read_csv(StringIO(r.text))
        print(f'  -> {len(df):,} rows fetched from remote.')
        return df
    except Exception as e:
        print(f'  ! remote fetch failed ({e}); falling back to local {fallback}')
        return pd.read_csv(fallback)

news = fetch_news(NEWS_URL, LOCAL_NEWS)
news.head(3)

## 1.3 Bug Fix 2 — Mixed date formats

The `date` column in `cryptonews.csv` mixes timezone-aware and timezone-naive strings (some rows end with `+00:00`, some don't). Using `format='mixed'` with `utc=True` normalizes everything to UTC.


In [ ]:
# Before
print('Raw date sample:', news['date'].iloc[0])
print('Mixed tz styles present:', news['date'].str.contains(r'[+\-]\d\d:\d\d$').sum(), 'aware /', (~news['date'].str.contains(r'[+\-]\d\d:\d\d$')).sum(), 'naive')

# Fix
news['date'] = pd.to_datetime(news['date'], format='mixed', utc=True, errors='coerce')
news = news.dropna(subset=['date']).sort_values('date').reset_index(drop=True)
print(f'After parse: {len(news):,} rows | range {news.date.min()} → {news.date.max()}')
news.head(3)

## 1.4 Parse the embedded sentiment dict

The `cryptonews.csv` ships with a TextBlob-style sentiment dict in the `sentiment` column. We extract `class`, `polarity`, `subjectivity` into separate columns. Notebook 02 will override these with LLM-derived scores.


In [ ]:
import ast

def parse_sentiment(s):
    try:
        d = ast.literal_eval(s) if isinstance(s, str) else {}
        return pd.Series({
            'sentiment_class': d.get('class', 'neutral'),
            'sentiment_polarity': float(d.get('polarity', 0.0)),
            'sentiment_subjectivity': float(d.get('subjectivity', 0.0)),
        })
    except Exception:
        return pd.Series({'sentiment_class': 'neutral', 'sentiment_polarity': 0.0, 'sentiment_subjectivity': 0.0})

sentiment_parsed = news['sentiment'].apply(parse_sentiment)
news = pd.concat([news.drop(columns=['sentiment']), sentiment_parsed], axis=1)
news[['date', 'title', 'sentiment_class', 'sentiment_polarity']].head(5)

## 1.5 Aggregate news to daily level

Multiple headlines publish per day. We aggregate to a daily level using mean polarity and headline count.


In [ ]:
news['date_day'] = news['date'].dt.tz_convert(None).dt.floor('D')
news_daily = (
    news.groupby('date_day')
         .agg(news_count=('title', 'size'),
              mean_polarity=('sentiment_polarity', 'mean'),
              mean_subjectivity=('sentiment_subjectivity', 'mean'),
              neg_share=('sentiment_class', lambda s: (s == 'negative').mean()),
              pos_share=('sentiment_class', lambda s: (s == 'positive').mean()))
         .reset_index()
         .rename(columns={'date_day': 'date'})
)
print(f'{len(news_daily):,} daily news rows | {news_daily.date.min()} → {news_daily.date.max()}')
news_daily.head(3)

## 1.6 Fetch BTC-USD via yfinance

### Bug Fix 1 — Flatten MultiIndex columns
Newer yfinance versions (≥0.2.31) return a `pd.MultiIndex` on columns when `auto_adjust=False`. The pipeline defensively flattens it.


In [ ]:
BTC_TICKER = 'BTC-USD'
btc = yf.download(BTC_TICKER, start='2023-01-01', end='2024-12-31', auto_adjust=False, progress=False)

print(f'Raw columns type: {type(btc.columns).__name__}')
print(f'Is MultiIndex  : {isinstance(btc.columns, pd.MultiIndex)}')

# ---- Bug Fix 1: flatten MultiIndex columns ----
if isinstance(btc.columns, pd.MultiIndex):
    btc.columns = [' '.join(c).strip() for c in btc.columns]

btc = btc.reset_index().rename(columns={'Date': 'date'})
# Normalize column names (strip ticker suffix)
btc.columns = [c.replace(' BTC-USD', '').lower() for c in btc.columns]
btc['date'] = pd.to_datetime(btc['date']).dt.floor('D')
print(f'BTC rows: {len(btc):,} | range {btc.date.min()} → {btc.date.max()}')
btc.tail(3)

## 1.7 Merge news + BTC prices


In [ ]:
merged = pd.merge(btc, news_daily, on='date', how='left')
merged[['news_count', 'mean_polarity', 'neg_share', 'pos_share']] = merged[
    ['news_count', 'mean_polarity', 'neg_share', 'pos_share']
].fillna(0)
merged = merged.sort_values('date').reset_index(drop=True)
print(f'Merged: {len(merged):,} rows | {merged.date.min()} → {merged.date.max()}')
merged.tail(5)

## 1.8 Persist interim artifact


In [ ]:
out_path = INTERIM / 'merged_daily.parquet'
merged.to_parquet(out_path, index=False)
print(f'Wrote {len(merged):,} rows → {out_path}')
print(f'File size: {out_path.stat().st_size / 1024:.1f} KB')

## 1.9 Summary
- Fetched news from `raw.githubusercontent.com` (no local uploads).
- Bug Fix 1 applied: yfinance MultiIndex columns flattened.
- Bug Fix 2 applied: mixed date formats parsed with `format='mixed', utc=True`.
- Daily news aggregated and merged with BTC OHLCV.
- Persisted to `notebooks/interim/merged_daily.parquet` for Notebook 02.
